In [12]:
#libaries
import pandas as pd
import numpy as np
import pycountry_convert as pc

In [25]:
df_long = pd.read_csv('../data/merged_data.csv')

In [ ]:
#COVID period flag
def covid_period(year):
  if year < 2020: return 'Pre-COVID'
  elif year == 2020: return 'COVID Peak'
  elif year <= 2022: return 'Recovery'
  else: return 'Post-COVID'

df_long['covid_period'] = df_long['year'].apply(covid_period)
print("✅ COVID period distribution:")
for period, count in df_long['covid_period'].value_counts().sort_index().items():
    print(f"   {period}: {count}")

COVID period: /n{'Pre-COVID': 4536, 'Recovery': 1512, 'Post-COVID': 1512, 'COVID Peak': 756}


In [23]:
# Gender unemployment gap
gender_pivot = df_long.pivot_table(
  index=['country_name', 'year', 'age_categories'],
  columns='sex',
  values='unemployment_rate'
).reset_index()

gender_pivot['gender_gap'] = gender_pivot['Female'] - gender_pivot['Male']
gender_pivot['gender_gap_abs'] = gender_pivot['gender_gap'].abs()
print("✅ Gender gap:")
print(f"   Average: {gender_pivot['gender_gap'].mean():.2f} percentage points")
print(f"   Shape: {gender_pivot.shape}")


Gender gap dataset shape: (4158, 7)

Average gender gap by age category:
                mean    min    max
age_categories                    
Adults          1.74  -6.64  19.97
Youth           3.55 -14.56  40.96

Overall average: 2.64 percentage points


In [ ]:
#Youth-to-adult unemployment ratio
youth = df_long[df_long['age_categories']=='Youth'][['country_name','year','sex','unemployment_rate']]
adult = df_long[df_long['age_categories']=='Adults'][['country_name','year','sex','unemployment_rate']]

ratio_df = youth.merge(adult, on=['country_name','year','sex'], suffixes=('_youth','_adult'))
ratio_df['youth_adult_ratio'] = ratio_df['unemployment_rate_youth'] / (ratio_df['unemployment_rate_adult'] + 0.01)
print("✅ Youth-adult ratio:")
print(f"   Average: {ratio_df['youth_adult_ratio'].mean():.2f}x")
print(f"   Youth > 2x adult: {(ratio_df['youth_adult_ratio']>2).mean()*100:.0f}%")
print(f"   Shape: {ratio_df.shape}")

In [17]:
#Economic development tier
df_long['dev_tier'] = pd.cut(
  df_long['gdp_per_capita_current_usd'],
  bins=[0, 1085, 4255, 13205, float('inf')],
  labels=['Low Income', 'Lower-Middle', 'Upper-Middle', 'High Income']
)
print("✅ Development tier distribution:")
for tier, count in df_long['dev_tier'].value_counts().sort_index().items():
    print(f"   {tier}: {count}")

ECONOMIC DEVELOPMENT TIER

World Bank Income Classification:
  Low Income: < $1,085
  Lower-Middle: $1,085 - $4,255
  Upper-Middle: $4,255 - $13,205
  High Income: > $13,205

Distribution:
dev_tier
Low Income      1056
Lower-Middle    2064
Upper-Middle    1956
High Income     2448
Name: count, dtype: int64

Percentage:
dev_tier
Low Income      14.0
Lower-Middle    27.4
Upper-Middle    26.0
High Income     32.5
Name: proportion, dtype: float64

Missing: 792

✅ Development tier feature created


In [18]:
# Year-over-year unemployment change
df_long = df_long.sort_values(['country_name','sex','age_categories','year'])
df_long['unemp_yoy_change'] = df_long.groupby(
  ['country_name','sex','age_categories']
)['unemployment_rate'].diff()
df_long['improving'] = (df_long['unemp_yoy_change'] < 0).astype(int)
print("✅ Year-over-year change:")
print(f"   Average: {df_long['unemp_yoy_change'].mean():.2f} percentage points")
print(f"   Improving: {df_long['improving'].mean()*100:.0f}%")

YEAR-OVER-YEAR UNEMPLOYMENT CHANGE

Average YoY change: -0.141 percentage points

Improving observations: 4200 (50.5%)
Worsening observations: 4116 (49.5%)

Largest increases (top 10):
    country_name  year    sex age_categories  unemp_yoy_change
            Iraq  2017 Female          Youth            25.071
          Panama  2020 Female          Youth            21.863
        Suriname  2016 Female          Youth            16.582
      Montenegro  2020 Female          Youth            15.612
   New Caledonia  2020   Male          Youth            14.624
    Saudi Arabia  2017 Female          Youth            14.167
      Costa Rica  2020 Female          Youth            13.384
Egypt, Arab Rep.  2018 Female          Youth            12.901
   New Caledonia  2020 Female          Youth            12.808
          Bhutan  2020 Female          Youth            11.349

Largest decreases (top 10):
     country_name  year    sex age_categories  unemp_yoy_change
           Panama  2021 Femal

In [19]:
# Add continent
def get_continent(name):
  try:
    code = pc.country_name_to_country_alpha2(name, cn_name_format="default")
    cont_code = pc.country_alpha2_to_continent_code(code)
    return pc.convert_continent_code_to_continent_name(cont_code)
  except: return 'Unknown'

df_long['continent'] = df_long['country_name'].apply(get_continent)
print("✅ Continent distribution:")
for continent, count in df_long['continent'].value_counts().items():
    print(f"   {continent}: {count}")

CONTINENT MAPPING

Continent distribution:
continent
Africa           2112
Asia             1848
Europe           1672
Unknown           880
North America     836
South America     484
Oceania           484
Name: count, dtype: int64

Percentage:
continent
Africa           25.4
Asia             22.2
Europe           20.1
Unknown          10.6
North America    10.1
South America     5.8
Oceania           5.8
Name: proportion, dtype: float64

Unknown countries: 880

Countries needing manual mapping (20):
['Bosnia And Herzegovina', 'Channel Islands', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Egypt, Arab Rep.', 'Hong Kong SAR, China', 'Iran, Islamic Rep.', 'Korea, Dem. Rep.', 'Korea, Rep.', 'Lao PDR', 'Macao SAR, China', 'Saint Vincent And The Grenadines', 'Sao Tome And Principe', 'South America']

✅ Continent feature created


In [20]:
# Save enhanced datasets
df_long.to_csv('../data/merged_data_features.csv', index=False)
gender_pivot.to_csv('../data/gender_gap.csv', index=False)
ratio_df.to_csv('../data/youth_adult_ratio.csv', index=False)

print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print("Files saved:")
print(f"   merged_data_features.csv: {df_long.shape}")
print(f"   gender_gap.csv: {gender_pivot.shape}")
print(f"   youth_adult_ratio.csv: {ratio_df.shape}")

FEATURE ENGINEERING COMPLETE

Files saved:
  ✅ merged_data_features.csv ((8316, 21))
  ✅ gender_gap.csv ((4158, 7))
  ✅ youth_adult_ratio.csv ((4158, 6))

New features created:
  - covid_period
  - dev_tier
  - unemp_yoy_change
  - improving
  - continent
  - gender_gap (separate file)
  - youth_adult_ratio (separate file)
